In [ ]:
# find model 과 blending model 분리하고 rmlse 값은 train만 적용하기 12.06

In [ ]:
# 💠 전체 메모리 클린업 (start)

import gc, import sys

def clean_all_memory():
    """GPU/CPU 캐시, gc, 객체 정리까지 가능한 모든 메모리 정리 수행"""

    print("🧹 전체 메모리 클린 중...")

    # 1) gc 수집
    gc.collect()

    # 2) IPython 환경이면 변수 초기화
    try:
        from IPython import get_ipython
        ipy = get_ipython()
        if ipy:
            ipy.magic("reset -f")
    except Exception:
        pass

    # 3) 파이썬 모듈 캐시 정리 (optional)
    # 너무 과하면 import 비용 증가 → 기본적으로는 안 건드림
    # sys.modules.clear()  # 필요한 경우에만 사용

    # 4) Torch CUDA 메모리 정리
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

            # cudnn 캐시도 비우고 싶다면 (매우 드문 경우)
            try:
                torch.backends.cudnn.deterministic = False
                torch.backends.cudnn.benchmark = True
            except Exception:
                pass

            print(f"🟢 GPU 메모리 정리 완료 (사용 가능 GPU: {torch.cuda.get_device_name(0)})")
    except Exception:
        print("ℹ️ Torch/CUDA 사용 안함 or 로드 안됨")

    print("✅ 전체 메모리 정리 완료.")

# 실행
try:
    clean_all_memory()
except NameError:
    # 없으면 간단히 gc만
    import gc
    gc.collect()

# ===========================# 💠 전체 메모리 클린업 (end) # ===========================


🧹 전체 메모리 클린 중...
🟢 GPU 메모리 정리 완료 (사용 가능 GPU: NVIDIA GeForce GTX 1660 SUPER)
✅ 전체 메모리 정리 완료.


In [ ]:
# analyzer = MercariPyCaretAnalyzer9(
#     data_dir="../data",
#     results_dir="../results",
#     model_dir="../models",
#     images_dir="../images",
#     use_gpu=True
# )

# analyzer.preprocess_all_staged(
#     use_cache=True,
#     save_cache=True,
#     cols=["name", "item_description"],
#     undersample_frac=0.30,
#     param_dict={"undersample_frac": 0.30},
#     debug=True
# )

# analyzer.vectorize_text(method="tfidf")  # 또는 "bert", "fasttext" 등
# analyzer.setup_pycaret(fold=3, use_gpu=True, n_jobs=4)
# analyzer.find_best_model(use_kaggle_winners=True)
# analyzer.save_metrics()
# analyzer.predict_test()


In [5]:
analyzer = MercariPyCaretAnalyzer9(
    data_dir="../data",
    results_dir="../results",
    model_dir="../models",
    images_dir="../images",
    use_gpu=True
)


In [6]:

analyzer.preprocess_all_staged(
    use_cache=True,
    save_cache=True,
    cols=["name", "item_description"],
    undersample_frac=0.30,
    param_dict={"undersample_frac": 0.30},
    debug=True
)



🚀 preprocess_all_staged() 시작

⚡ Stage loaded: loaded <- ../results\stage_loaded.pkl (timestamp: 2025-12-05T18:22:32.891458)
Loaded stage 'loaded' from cache.
⚡ Stage loaded: normalized <- ../results\stage_normalized.pkl (timestamp: 2025-12-05T18:23:00.578680)
Stage 'normalized' loaded from cache - skipping.
⚡ Stage loaded: text_stats <- ../results\stage_text_stats.pkl (timestamp: 2025-12-05T18:27:57.188989)
Stage 'text_stats' loaded from cache - skipping.
⚡ Stage loaded: price_brand_cat <- ../results\stage_price_brand_cat.pkl (timestamp: 2025-12-05T18:28:54.419145)
Stage 'price_brand_cat' loaded from cache - skipping.
⚡ Stage loaded: interactions <- ../results\stage_interactions.pkl (timestamp: 2025-12-05T18:29:00.118424)
Stage 'interactions' loaded from cache - skipping.

✅ preprocess_all_staged complete


In [7]:
analyzer.vectorize_text(method="tfidf")  # 또는 "bert", "fasttext" 등


📂 벡터화 데이터 로드됨: ../models\vectorized_tfidf_train.pkl, ../models\vectorized_tfidf_test.pkl


In [8]:
analyzer.setup_pycaret(fold=3, use_gpu=True, n_jobs=4)

🔧 PyCaret setup 시작...
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce GTX 1660 SUPER, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score 0.500000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped trainin

In [9]:
analyzer.find_best_model(use_kaggle_winners=True)



🤖 모델 탐색 시작... (최종 기준: RMSLE(log1p))
📌 Kaggle winners 기반 후보 모델만 사용합니다.

   - 모델 생성: lightgbm


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.0153  0.0011  0.0328  0.9981  0.0075  0.0051
1     0.0159  0.0012  0.0342  0.9979  0.0078  0.0053
2     0.0159  0.0010  0.0321  0.9981  0.0076  0.0053
Mean  0.0157  0.0011  0.0330  0.9980  0.0076  0.0052
Std   0.0003  0.0001  0.0009  0.0001  0.0001  0.0001
📊 RMSLE(log1p) 계산 중 (train 전체 예측 기반)...
   → RMSLE(log1p) = 0.030068

   - 모델 생성: xgboost


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.0138  0.0012  0.0345  0.9979  0.0075  0.0044
1     0.0147  0.0014  0.0371  0.9975  0.0081  0.0047
2     0.0140  0.0012  0.0342  0.9979  0.0077  0.0046
Mean  0.0142  0.0012  0.0353  0.9978  0.0078  0.0046
Std   0.0004  0.0001  0.0013  0.0002  0.0002  0.0001
📊 RMSLE(log1p) 계산 중 (train 전체 예측 기반)...
   → RMSLE(log1p) = 0.027055

   - 모델 생성: catboost


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.0315  0.0061  0.0784  0.9889  0.0167  0.0104
1     0.0322  0.0063  0.0793  0.9887  0.0168  0.0106
2     0.0321  0.0060  0.0774  0.9892  0.0167  0.0106
Mean  0.0319  0.0061  0.0784  0.9889  0.0167  0.0105
Std   0.0003  0.0001  0.0008  0.0002  0.0000  0.0001
📊 RMSLE(log1p) 계산 중 (train 전체 예측 기반)...
   → RMSLE(log1p) = 0.066497

   - 모델 생성: et


         MAE     MSE    RMSE      R2   RMSLE   MAPE
Fold                                               
0     0.0057  0.0007  0.0258  0.9988  0.0064  0.002
1     0.0060  0.0008  0.0291  0.9985  0.0069  0.002
2     0.0058  0.0007  0.0264  0.9987  0.0064  0.002
Mean  0.0058  0.0007  0.0271  0.9987  0.0065  0.002
Std   0.0001  0.0001  0.0014  0.0001  0.0003  0.000
📊 RMSLE(log1p) 계산 중 (train 전체 예측 기반)...
   → RMSLE(log1p) = 0.013314

   - 모델 생성: rf


Processing:  25%|██▌       | 1/4 [00:01<00:05,  1.87s/it]

KeyboardInterrupt: 

In [ ]:
analyzer.save_metrics()


In [ ]:
analyzer.predict_test()
